Charger le fichier products.csv. En utilisant pyspark, repérer les valeurs aberrantes (ordre de grandeur : plusieurs centaines).

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab4App") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
products_df = spark.read.csv("../../data/products.csv", header=True, inferSchema=True)

In [9]:
from pyspark.sql.functions import col

tolerance_2 = 1 

col_to_sum = [
    "fat_100g",
    "sugars_100g",
]

products_df.withColumn(
    "total_313",
    sum(col(col_name) for col_name in col_to_sum)
).filter(
    col("total_313") < 100 - tolerance_2
).show()

+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+------------------+
|_c0|   code|fat_100g|saturated-fat_100g|sugars_100g|fiber_100g|proteins_100g|salt_100g|sodium_100g|             autre|         total_313|
+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+------------------+
|  1| 4530.0|   28.57|             28.57|      14.29|       3.6|         3.57|      0.0|        0.0|21.400000000000006|             42.86|
|  2| 4559.0|   17.86|               0.0|      17.86|       7.1|        17.86|    0.635|       0.25|            38.435|             35.72|
|  3|16087.0|   57.14|              5.36|       3.57|       7.1|        17.86|  1.22428|      0.482| 7.263720000000021|             60.71|
|  5|16100.0|   18.27|              1.92|      11.54|       7.7|        13.46|     NULL|       NULL|             47.11|             29.81|
|  7|16124.0|   18.75|     

In [10]:
tolerance_2 = 1

products_df.createOrReplaceTempView("products")

col_to_sum = [
    "fat_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "autre"
]

sum_expr = " + ".join(col_to_sum)

query = f"""
SELECT *,
       ({sum_expr}) AS new_total
FROM products
WHERE ({sum_expr}) < {100 - tolerance_2}
"""

result_df = spark.sql(query)

print(result_df.count())
result_df.show()

107348
+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+-----------------+
|_c0|   code|fat_100g|saturated-fat_100g|sugars_100g|fiber_100g|proteins_100g|salt_100g|sodium_100g|             autre|        new_total|
+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+-----------------+
|  1| 4530.0|   28.57|             28.57|      14.29|       3.6|         3.57|      0.0|        0.0|21.400000000000006|            71.43|
|  3|16087.0|   57.14|              5.36|       3.57|       7.1|        17.86|  1.22428|      0.482| 7.263720000000021|94.15800000000002|
|  7|16124.0|   18.75|              4.69|      15.62|       9.4|        14.06|   0.1397|      0.055|           37.2853|           95.255|
| 12|16872.0|   36.67|               5.0|       3.33|       6.7|        16.67|  1.60782|      0.633|29.389179999999996|           94.367|
| 15|18012.0|   18.18|     

In [11]:
query = f"""
SELECT *,
       ({sum_expr}) AS new_total
FROM products
WHERE ({sum_expr}) < (100 - :tolerance)
"""

result_df = spark.sql(
    query,
    {"tolerance": tolerance_2}
)

result_df.show()

+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+-----------------+
|_c0|   code|fat_100g|saturated-fat_100g|sugars_100g|fiber_100g|proteins_100g|salt_100g|sodium_100g|             autre|        new_total|
+---+-------+--------+------------------+-----------+----------+-------------+---------+-----------+------------------+-----------------+
|  1| 4530.0|   28.57|             28.57|      14.29|       3.6|         3.57|      0.0|        0.0|21.400000000000006|            71.43|
|  3|16087.0|   57.14|              5.36|       3.57|       7.1|        17.86|  1.22428|      0.482| 7.263720000000021|94.15800000000002|
|  7|16124.0|   18.75|              4.69|      15.62|       9.4|        14.06|   0.1397|      0.055|           37.2853|           95.255|
| 12|16872.0|   36.67|               5.0|       3.33|       6.7|        16.67|  1.60782|      0.633|29.389179999999996|           94.367|
| 15|18012.0|   18.18|            

In [20]:
spark.stop()